# Phase 3: Validation strategy
The 3 test subjects are unseen, so the leaderboard measures **generalisation to new people**. Shuffled k-fold would put the same person in train and validation and look far too optimistic.

**What your outputs showed:** `subject_id` is not given, rows are shuffled (label change 0.922 vs 0.913 expected) and the ids are shuffled too (re-checked: sorting by id number gives 0.915 vs 0.913, and neighbouring ids are no more similar than random pairs). So order cannot reveal subjects.

**What we do instead:** recover pseudo-subjects from the data. Windows of the same person share sensor placement and body offsets, so within one activity the windows form clusters. I checked on your data: within sitting / lying / standing the window-median signature gives 5 very clean clusters (silhouette peaks at k=5: 0.75, 0.68, 0.55), matching the 5 training subjects. This notebook clusters **inside each class** into 5 groups and uses the cluster id as the validation fold. Each fold then holds out one (pseudo-)subject's windows per class.

Limitation to state in the report: clusters are inferred, not true IDs, and cluster number is not aligned across classes, so the same person can sit in different folds for different activities.

Input `clean_data_v2.npz`. Output `splits.npz`.

In [1]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

d = np.load("clean_data_v2.npz", allow_pickle=False)
X, y = d["X_train"], d["y_train"]
classes = sorted(np.unique(y))
N_SUBJ = 5
print(X.shape, len(classes), "classes")

(635, 250, 23) 12 classes


## 1. Does cluster structure exist per class?
Signature = per-channel median of each window (captures posture, sensor offsets, mag orientation). Silhouette over k=2..7 should peak at 5 if windows split by subject. Classes where it does not peak at 5 are still split into 5 clusters, but treat them with more caution.

In [2]:
sig = StandardScaler().fit_transform(np.median(X, axis=1))   # (n, 23)
rows = []
for c in classes:
    F = sig[y == c]
    sil = {k: silhouette_score(F, KMeans(k, n_init=10, random_state=0).fit_predict(F)) for k in range(2, 8) if k < len(F)}
    rows.append({"class": c, "n": len(F), "best_k": max(sil, key=sil.get), "sil_at_5": round(sil.get(5, np.nan), 2)})
print(pd.DataFrame(rows).to_string(index=False))

          class  n  best_k  sil_at_5
  arm_elevation 54       6      0.32
climbing_stairs 54       7      0.27
        cycling 59       5      0.58
        jogging 59       2      0.27
jump_front_back 17       3      0.34
  knees_bending 50       7      0.43
          lying 60       5      0.95
        running 58       2      0.25
        sitting 58       5      0.96
       standing 58       5      0.85
    waist_bends 49       7      0.42
        walking 59       2      0.46


## 2. Build folds
Fold = cluster id within the class. `jump_front_back` has only 17 windows so its clusters are tiny (3-4 each); that is fine, every fold still sees it.

In [3]:
folds = np.zeros(len(y), int)
for c in classes:
    idx = np.where(y == c)[0]
    F = sig[idx]
    lab = KMeans(N_SUBJ, n_init=20, random_state=0).fit_predict(F)
    # order clusters by size so fold numbers are stable and sizes are balanced across classes
    rank = {k: r for r, k in enumerate(pd.Series(lab).value_counts().index)}
    folds[idx] = [rank[l] for l in lab]
ct = pd.crosstab(y, folds); print(ct)
print("\nwindows per fold:", np.bincount(folds))
assert (ct.values > 0).all(), "some class is missing from a fold"
np.savez("splits.npz", folds=folds, scheme="pseudo_subject_per_class")

col_0             0   1   2   3   4
row_0                              
arm_elevation    17  12  11   8   6
climbing_stairs  18  11  10   8   7
cycling          13  12  12  11  11
jogging          18  17  12   6   6
jump_front_back   7   4   3   2   1
knees_bending    11  11  10   9   9
lying            12  12  12  12  12
running          20  18  11   5   4
sitting          12  12  12  11  11
standing         12  12  12  11  11
waist_bends      17  13   9   8   2
walking          23  12  11  11   2

windows per fold: [180 146 125 102  82]


## 3. Sanity check: is this validation harder than random k-fold?
A quick logistic-regression baseline on simple window means/stds with both schemes. If the pseudo-subject accuracy is clearly **lower** than random k-fold, the folds are capturing a subject effect (which is what we want: it mimics the test situation).

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
F = np.hstack([X.mean(1), X.std(1)])
mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
def cv(fold_ids):
    acc = []
    for k in np.unique(fold_ids):
        tr, va = fold_ids != k, fold_ids == k
        acc.append((mk().fit(F[tr], y[tr]).predict(F[va]) == y[va]).mean())
    return np.mean(acc), np.round(acc, 3)
rand = np.zeros(len(y), int)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(F, y)): rand[va] = k
print("random stratified 5-fold:", cv(rand))
print("pseudo-subject 5-fold   :", cv(folds))

random stratified 5-fold: (np.float64(0.9858267716535434), array([0.976, 1.   , 0.976, 0.984, 0.992]))
pseudo-subject 5-fold   : (np.float64(0.8118530034131929), array([0.883, 0.904, 0.68 , 0.824, 0.768]))
